# Group 25 - Evaluation Notebook
# Krones Bottle-Base Inspection -- Inference + Runtime Measurement


## 0 - Imports

In [ ]:
import os, sys, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import cv2

try:
    import onnxruntime as ort
except ImportError:
    os.system(f"{sys.executable} -m pip install -q onnxruntime-gpu"); import onnxruntime as ort

print("onnxruntime version:", ort.__version__)

## 1 - Configuration



In [ ]:
cfg = {
    # ---- paths ----
    "model_dir": "/kaggle/input/group-xx-bottle-inspection-model",  # <-- attached dataset path
    "data_dir":  "/kaggle/input/competitions/1st-krones-vision-ai-challenge",
    "onnx_filename": "bottle_inspection_ensemble.onnx",

    # ---- preprocessing (MUST match the training notebook's cfg) ----
    "image_size": 448,
    "input_channels": 1,
    "normalize_mean": 0.449,
    "normalize_std": 0.226,
    "roi_margin": 0.06,

    # ---- decision (copy from the training notebook's printed values) ----
    "decision_threshold": 0.3025,                 # <-- paste final_threshold from training
    "fallback_roi": [160.0, 147.0, 987.0, 722.0],# <-- paste fallback_roi from training

    # ---- inference ----
    "batch_size": 16,
}

model_dir = Path(cfg["model_dir"])
data_dir = Path(cfg["data_dir"])
test_image_dir = data_dir / "test_images"
test_annotations_json = data_dir / "test_annotations_roi_only.json"
print("Model dir exists:", model_dir.exists(), "| Test images dir exists:", test_image_dir.exists())

## 2 - Load the ONNX model
Request GPU, fall back to CPU, and print the active provider (it should show CUDA — important for the runtime measurement).

In [ ]:
onnx_session = ort.InferenceSession(
    str(model_dir / cfg["onnx_filename"]),
    providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
print("Active execution providers:", onnx_session.get_providers())
onnx_input_name = onnx_session.get_inputs()[0].name

## 3 - ROI crop (identical to training)

In [ ]:
def crop_to_roi(image, roi_x, roi_y, roi_w, roi_h, margin, output_size):
    """Square crop around the ROI box (+margin), reflect-pad if needed, resize to output_size."""
    height, width = image.shape[:2]
    center_x, center_y = roi_x + roi_w / 2.0, roi_y + roi_h / 2.0
    side = max(roi_w, roi_h) * (1.0 + 2.0 * margin)
    left, top = int(round(center_x - side / 2)), int(round(center_y - side / 2))
    right, bottom = int(round(center_x + side / 2)), int(round(center_y + side / 2))
    pad_left, pad_top = max(0, -left), max(0, -top)
    pad_right, pad_bottom = max(0, right - width), max(0, bottom - height)
    cropped = image[max(0, top):min(height, bottom), max(0, left):min(width, right)]
    if pad_left or pad_top or pad_right or pad_bottom:
        cropped = cv2.copyMakeBorder(cropped, pad_top, pad_bottom, pad_left, pad_right,
                                     cv2.BORDER_REFLECT_101)
    interpolation = cv2.INTER_AREA if cropped.shape[0] > output_size else cv2.INTER_LINEAR
    return cv2.resize(cropped, (output_size, output_size), interpolation=interpolation)

## 4 - Test image list + ROI map
The test set ships ROI-only annotations. We list only real image files from the directory.

In [ ]:
roi_box_by_filename = {}
if test_annotations_json.exists():
    with open(test_annotations_json) as f:
        test_coco = json.load(f)
    test_image_id_to_filename = {img["id"]: Path(img["file_name"]).name
                                 for img in test_coco["images"]}
    roi_box_by_filename = {test_image_id_to_filename[ann["image_id"]]: ann["bbox"]
                           for ann in test_coco["annotations"]}

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
def sort_key(filename):
    stem_tail = Path(filename).stem.split("_")[-1]
    return int(stem_tail) if stem_tail.isdigit() else filename
test_filenames = sorted(
    [p.name for p in test_image_dir.iterdir()
     if p.is_file() and p.suffix.lower() in image_extensions],
    key=sort_key)
fallback_roi = tuple(cfg["fallback_roi"])
print("Test images:", len(test_filenames), "| ROI boxes available:", len(roi_box_by_filename))

## 5 - Preprocess + run the model
The ONNX ensemble already outputs a FAULTY probability (sigmoid + fold-averaging happen inside the graph), so we use the output directly with no further transformation.

In [ ]:
def preprocess_image(filename):
    """Load grayscale, crop to ROI, normalize -> (1, H, W) float32 (matches training)."""
    image = cv2.imread(str(test_image_dir / filename), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise FileNotFoundError(test_image_dir / filename)
    roi_x, roi_y, roi_w, roi_h = roi_box_by_filename.get(filename, fallback_roi)
    image = crop_to_roi(image, roi_x, roi_y, roi_w, roi_h, cfg["roi_margin"], cfg["image_size"])
    array = image.astype(np.float32) / 255.0
    array = (array - cfg["normalize_mean"]) / cfg["normalize_std"]
    return array[np.newaxis, :, :]

def run_inference(filenames):
    """Return the FAULTY probability for every filename, processed in batches."""
    all_probabilities = np.zeros(len(filenames), dtype=np.float32)
    batch_size = cfg["batch_size"]
    for start in range(0, len(filenames), batch_size):
        batch_filenames = filenames[start:start + batch_size]
        batch = np.stack([preprocess_image(name) for name in batch_filenames]).astype(np.float32)
        batch_output = onnx_session.run(None, {onnx_input_name: batch})[0]
        all_probabilities[start:start + len(batch_filenames)] = np.asarray(batch_output).reshape(-1)
    return all_probabilities

## 6 - Predict and write submission.csv

In [ ]:
faulty_probabilities = run_inference(test_filenames)
predictions = (faulty_probabilities >= cfg["decision_threshold"]).astype(int)

submission = pd.DataFrame({"image_id": test_filenames, "target": predictions})
submission.to_csv("submission.csv", index=False)

num_faulty = int(submission["target"].sum())
print(f"submission.csv: {len(submission)} rows | FAULTY = {num_faulty} "
      f"({100*num_faulty/len(submission):.1f}%)")
print(submission.head())

## 7 - Inference runtime measurement (efficiency metric)
Times the full pipeline (load + crop + preprocess + model) over the whole test set after a warmup pass.

In [ ]:
_ = run_inference(test_filenames[:min(64, len(test_filenames))])   # warmup, not timed

start_time = time.perf_counter()
_ = run_inference(test_filenames)
total_seconds = time.perf_counter() - start_time

num_images = len(test_filenames)
required_throughput = 70000 / 3600
measured_throughput = num_images / total_seconds
print("=" * 60)
print(f"INFERENCE RUNTIME ({num_images} images)")
print("=" * 60)
print(f"Total time : {total_seconds:.2f} s")
print(f"Per image  : {total_seconds/num_images*1000:.2f} ms")
print(f"Throughput : {measured_throughput:.1f} images/sec")
print(f"Line speed : {required_throughput:.1f} images/sec required (70,000 bottles/hour)")
print(f"Headroom   : {measured_throughput/required_throughput:.2f}x "
      f"{'PASS' if measured_throughput >= required_throughput else 'BELOW'}")

## 8 - Done

`submission.csv` is written and the runtime is printed above. The notebook ran end to end with no
manual steps, loading the model from the attached (shared) dataset.